# Reproducibility: 原始输出只读交付 + P1 CPU 复核

选择 **CPU → 全部运行**；无需 T4/A100、GH_TOKEN 或 HF_TOKEN。
不训练、不推理、不改原始归档、论文、指标或阈值。代码已嵌入并有 SHA-256 校验。

本 notebook 读取 Drive 中已完成的 P1、Transparent Reference 和（可选）P3 原始输出，
先打包保存，再从 P1 原始响应重算四项指标及原 5,000 次 bootstrap，逐项对照历史结果。
CPU 复算需要一些时间，请等最后的下载单元；不必重新跑模型。

**生成的 ZIP 是给作者核对的私有交接包，不是已完成匿名审查的投稿材料。**
其中 P3 NPZ 含第三方 target 片段和原 series IDs：不要上传公开 GitHub / OpenReview。
本 notebook 不打包原始 Parquet、Token、模型权重、日志或其他 Drive 文件。
SQL 只能重新汇总已保存的损失，不能从缺失的 quantile forecasts 重算；不会伪装成已解决。

跑完把 `export_report.json` 和 `primary_replay_report.json` 发来；ZIP 通过私有附件交接。
如果复核报错，原始文件和已导出的 ZIP 都会保留，复制实际错误，不要修改阈值。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import subprocess, sys, os, json, hashlib, base64, io, zipfile, tempfile

PROJECT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness')
# True = 仅在私有交接包中包含 P3 的全部十二个 cells；不是公开发布许可。
INCLUDE_P3_PRIVATE = True
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
EXPORT = PROJECT / 'reproducibility_exports' / stamp
REPLAY = EXPORT.parent / (EXPORT.name + '_primary_replay')
print('Read-only sources:', PROJECT)
print('New private output:', EXPORT)


In [ ]:
# Embedded code/config/public summaries only; no private model outputs.
PAYLOAD_SHA256 = '9357428ffe431f2450caa0a00fea2de8b548a41265bb07c3d69fe74c473990b7'
PAYLOAD_BASE64 = '...'
payload = base64.b64decode(PAYLOAD_BASE64, validate=True)
assert hashlib.sha256(payload).hexdigest() == PAYLOAD_SHA256, 'Embedded snapshot integrity failure'
ROOT = Path(tempfile.mkdtemp(prefix='covshape_handoff_'))
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for member in archive.infolist():
        target = (ROOT / member.filename).resolve()
        if not target.is_relative_to(ROOT) or member.is_dir():
            raise RuntimeError('Unsafe embedded snapshot member')
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(member))
snapshot = json.loads((ROOT / 'reproducibility/export_snapshot.json').read_text())
for relative, expected in snapshot['files'].items():
    assert hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == expected, relative
# Dependencies stay in this temporary runtime, not the Colab global environment.
DEPS = ROOT / '_cpu_deps'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', str(DEPS), '-r',
                str(ROOT / 'reproducibility/requirements-cpu.txt')], check=True)
ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(DEPS)
ENV['PYTHONNOUSERSITE'] = '1'
print('Embedded snapshot verified:', PAYLOAD_SHA256)


In [ ]:
command = [sys.executable, str(ROOT / 'reproducibility/export_archives.py'),
           '--repo', str(ROOT), '--project', str(PROJECT), '--output', str(EXPORT)]
if INCLUDE_P3_PRIVATE:
    command.append('--include-p3-private')
subprocess.run(command, cwd=ROOT, env=ENV, check=True)
report = json.loads((EXPORT / 'export_report.json').read_text())
print(json.dumps({name: {k: v for k, v in info.items() if k != 'npz_schema'}
                  for name, info in report['groups'].items()}, indent=2))
print('Private handoff ZIP saved before numerical replay:', EXPORT / 'covshape_author_handoff.zip')


In [ ]:
# Reads original response arrays only. Writes recomputation to a separate new directory.
command = [sys.executable, str(ROOT / 'reproducibility/verify_primary_responses.py'),
           '--repo', str(ROOT), '--source', str(PROJECT / 'p1_shape_v1'), '--output', str(REPLAY)]
completed = subprocess.run(command, cwd=ROOT, env=ENV)
if completed.returncode:
    print('CPU replay failed. The handoff ZIP is still available; copy the actual error above.')
else:
    replay = json.loads((REPLAY / 'primary_replay_report.json').read_text())
    fields = ('result_status', 'scope', 'evidence_counts')
    print(json.dumps({k: replay[k] for k in fields}, indent=2))


In [ ]:
from google.colab import files
print('Private handoff:', EXPORT / 'covshape_author_handoff.zip')
files.download(str(EXPORT / 'covshape_author_handoff.zip'))
files.download(str(EXPORT / 'export_report.json'))
if (REPLAY / 'primary_replay_report.json').is_file():
    files.download(str(REPLAY / 'primary_replay_report.json'))
    files.download(str(REPLAY / 'primary_recomputed.csv'))
print('若浏览器拦截多个下载，请从上面显示的 Drive 路径手动下载。不要公开发布交接包。')
